In [81]:
import pandas as pd
import numpy as np
from combat.pycombat import pycombat  # pip install combat
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import plotly.express as px
import plotly.graph_objects as go  # For heatmaps

In [64]:
# To make plotly fig show in notebook
import plotly.io as pio
pio.renderers.default = "notebook"

# MAIN

In [65]:
# Read filtered TSV (only non fully '0' rows)
X = pd.read_csv("CUSTOM_hg38_episign/meth_matrix.tsv", sep="\t", index_col=0)

# Compute epiSize (= size of epiSign):
epiSize = {}
for epiSign in [x for x in X.columns if x not in ('coord')]:
    epiSize[epiSign] = sum(X[epiSign] > 0)  # Catch 100% methyl

## Pre-processing

### Batch-effect correction using (py)combat

In [66]:
# First we generate the list of batches:
ref_sign = ["ADCADN","ATRX","AUTS18","BAFopathy","BFLS","CHARGE","CdLS","Down","Dup7","EEOC","FLHS","GTPTS","HMA","HVDAS_C","HVDAS_T","ICF1","ICF2_3_4","KDVS","Kabuki","Kleefstra","MRD51","MRX93","MRX97","MRXCJS","MRXSN","MRXSSR","RMNS","RSTS","SBBYSS","SETD1B","Sotos","TBRS","WDSTS","Williams"]
dataset_251217 = ["HG002_combined","barcode04_combined"]

batch = []
datasets = [ref_sign, dataset_251217]
for j in range(len(datasets)):
    batch.extend([j for _ in range(len(datasets[j]))])

# Then run (py)combat:
X_corrected = pycombat(X, batch)

Found 2 batches.
Adjusting for 0 covariate(s) or covariate level(s).
Standardizing Data across genes.
Fitting L/S model and finding priors.
Finding parametric adjustments.
Adjusting the Data


/home/felix/.local/share/mamba/envs/NSBEpi/lib/python3.9/site-packages/combat/pycombat.py:159: RuntimeWarning: divide by zero encountered in divide
  np.absolute(d_new-d_old)/d_old))  # maximum difference between new and old estimate


### Transpose + normalize

In [139]:
USE_COMBAT = False
USE_NORMALIZED = False
EPISIGN_OF_INTEREST = ['RMNS','Kleefstra','Kabuki','barcode04_combined','HG002_combined']

# Transpose (required):
X_t = X.T
if USE_COMBAT:
    X_t = X_corrected.T

# Remove 2nd row = size of epiSign then normalize
to_norm = X_t
if 'epiSize' in X_t.columns:
    to_norm = X_t.drop('epiSize', axis=1)
print(to_norm[to_norm.columns[0:3]].head())

# WARN: If norm, should be AFTER combat
to_PCA = to_norm
if USE_NORMALIZED:
    to_PCA = pd.DataFrame(StandardScaler().fit_transform(to_norm), columns=to_norm.columns, index=to_norm.index)

# Write corrected file:
for sample in ['barcode04_combined', 'HG002_combined']:
    #to_PCA.T[sample].to_csv(f"{sample}_corrected.tsv", sep="\t")
    print(f" > Wrote: {sample}_corrected.tsv")

coord      1:21345014-21345015  1:47416641-47416642  1:47444566-47444567
ADCADN                 0.63112             0.341521             0.292684
ATRX                   0.00000             0.000000             0.000000
AUTS18                 0.00000             0.000000             0.000000
BAFopathy              0.00000             0.000000             0.000000
BFLS                   0.00000             0.000000             0.000000
 > Wrote: barcode04_combined_corrected.tsv
 > Wrote: HG002_combined_corrected.tsv


## Heatmaps

In [140]:
# MEMO: With index, plot broken with 'px.imshow'
names = to_PCA.index
coord = to_PCA.columns

go.Figure(data=go.Heatmap(
    z = to_PCA.to_numpy(),
    x = coord,
    y = names
))

In [141]:
# MEMO: Have to drop index, otherwise plot broken
# Width/height empirical bellow
fig = px.imshow(
    to_PCA.to_numpy(),
    y = names,
    x = coord,
    width = 15000,
    height = 1000
)
fig.update_layout(
    autosize=False,
    xaxis=dict(tickfont=dict(size=5)),
    yaxis=dict(tickfont=dict(size=5))
)

## PCA

In [142]:
# Run PCA:
NB_COMPON = 3
pca = PCA(n_components=NB_COMPON)
pcs = pca.fit_transform(to_PCA.to_numpy())

# Make a dict with '% variance explained' for each component:
dict_compon = {'compon'+str(i) : str(round(pca.explained_variance_ratio_[i]*100,4)) for i in range(NB_COMPON)}

In [143]:
# Top N features of each componennt
compon_0_top = np.abs(pca.components_[0]).argsort()[::-1][:5]
print("Component 0:", list(X.index[compon_0_top]))

compon_1_top = np.abs(pca.components_[1]).argsort()[::-1][:5]
print("Component 1:", list(X.index[compon_1_top]))

Component 0: ['7:102300990-102300991', '5:13810085-13810086', '7:149713831-149713832', '19:41844499-41844500', '16:15694082-15694083']
Component 1: ['1:175406891-175406892', '10:122149940-122149941', '22:37675583-37675584', '1:3688522-3688523', '2:30111451-30111452']


In [144]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
pcs_df = pd.DataFrame(
    pcs,
    index=X.columns,
    columns=dict_compon.keys()
)
print(pcs_df.loc[EPISIGN_OF_INTEREST])

                      compon0    compon1   compon2
RMNS                -1.676756  -0.242586 -0.064830
Kleefstra           -1.494633   0.183185  0.003870
Kabuki              -1.401136   0.175430 -0.421736
barcode04_combined  25.563672  10.349041 -0.216865
HG002_combined      27.259122  -9.766160  0.074982


In [145]:
# Plot PCA
color_selected = [ x in SIGN_OF_INTEREST for x in pcs_df.index ]

x_compon = 'compon0'
y_compon = 'compon1'

fig = px.scatter(
        pcs_df,
        x=x_compon,
        y=y_compon,
        hover_data=[pcs_df.index],
        color=color_selected,
        labels={x_compon:':'.join([x_compon,"", dict_compon[x_compon]]), y_compon:':'.join([y_compon,"",dict_compon[y_compon]])}
)
fig.show()

## t-SNE

In [146]:
# Run t-SNE:
# MEMOs:
# - In Joris' paper they use 'preplex=2'
# - t-SNE is stochastic -> re-run multiple times ?
#
tsne = TSNE(n_components=2, perplexity=2).fit_transform(to_PCA)

In [147]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
tsne_df = pd.DataFrame(
    tsne,
    index=X.columns,
    columns=('compon0', 'compon1')
)
subset_tsne = tsne_df.loc[['Kabuki', 'barcode04_combined', 'HG002_combined']]
print(subset_tsne)

                       compon0     compon1
Kabuki             -104.077484   24.786459
barcode04_combined -198.473831 -167.679535
HG002_combined     -187.241104 -171.177246


In [148]:
# Plot t-SNE
color_selected = [x in EPISIGN_OF_INTEREST for x in tsne_df.index]
fig = px.scatter(
        tsne_df,
        x='compon0',
        y='compon1',
        hover_data=[tsne_df.index],
        color=color_selected
)
fig.show()